# 29c. Inference Observability: Prometheus & Grafana

**Tier:** Production & Safety
**Estimated time:** 50 minutes
**Prerequisites:** 27 (production monitoring), 29b (load testing inference)
**Priority:** 🟡 Important — the infra layer underneath the quality-focused monitoring in notebook 27. *If skipped, revisit when:* you need a real dashboard your on-call rotation can page off of, not just an eval script you run by hand.
**Source material:** The Prometheus exposition format and `histogram_quantile()` semantics (Prometheus docs); `prometheus_client` (the official Python client library); Grafana's dashboard-as-JSON model.

## What You'll Learn
- How to instrument a real server with the three Prometheus metric types that matter for inference — **Counter**, **Histogram**, **Gauge** — and why the choice between them isn't arbitrary
- Why the *default* histogram buckets almost every tutorial ships with are wrong for LLM latencies, and what breaks when you use them anyway
- How to scrape `/metrics` and compute `rate()` and `histogram_quantile()` **by hand**, so PromQL stops being a black box
- How to turn a token counter into a live **$/hour** figure, and ship the `docker-compose.yml` + dashboard JSON that would make all of this real on your own machine

## Why This Matters
Notebook 27's production monitoring watches *quality* — is the model still answering well, has a prompt change regressed something. This notebook watches the *infrastructure* underneath: is the GPU saturated, is p99 latency creeping up, is this feature quietly costing more per hour than it did yesterday. Those are different failure modes with a different tool built for exactly them, and it's the tool nearly every real inference stack (vLLM, SGLang, and any hand-rolled server like notebook 29's) exposes a `/metrics` endpoint for.


## The three metric types, and why the choice matters

- **Counter** — a number that only ever goes up (total requests, total tokens, total errors). You never read a counter's raw value; you compute its **rate** — how fast it's climbing — because the raw value only tells you "how long has this process been running," not "how loaded is it right now."
- **Histogram** — buckets observations (like latencies) into `≤ X` counts, so you can approximate any percentile later without having stored every individual value. The **bucket boundaries you pick are a real design decision** — pick them wrong and you silently lose the ability to distinguish a merely-late request from a catastrophically slow one.
- **Gauge** — a value that goes up AND down (queue depth, active GPU slots). Unlike a counter, you read a gauge's current value directly — there's no "rate of queue depth" that means anything.

Getting the type wrong is a common real mistake: making queue depth a Counter (it can't decrease, so the metric can only tell you the same lie a resettable odometer would) or making request count a Gauge (You'd lose the ability to compute rate() across a restart-safe running total).


In [1]:
import os, sys, time, socket, subprocess, atexit, textwrap
from pathlib import Path

def find_free_port():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]

PORT = find_free_port()
APP_DIR = Path("/tmp/nb29c_app")
APP_DIR.mkdir(exist_ok=True)
print(f"Will run the instrumented mock server on port {PORT}")


Will run the instrumented mock server on port 45293


## Instrumenting the server

Same simulated-GPU mock server as notebook 29b (a semaphore standing in for GPU capacity), now wired up with real `prometheus_client` instruments at every point that matters: a **Counter** per completed request and per token processed, a **Histogram** for TTFT and end-to-end latency, and a **Gauge** for how many requests are currently queued vs. actively running.


In [2]:
APP_SOURCE = textwrap.dedent(r'''
    import asyncio, random, time
    from fastapi import FastAPI, Response
    from pydantic import BaseModel
    from prometheus_client import Counter, Histogram, Gauge, generate_latest, CONTENT_TYPE_LATEST

    app = FastAPI()

    BATCH_SLOTS = 16
    TTFT_BASE_S = 0.03
    TPOT_S = 0.012
    N_TOKENS = 8

    gpu_slots = asyncio.Semaphore(BATCH_SLOTS)

    REQUESTS_TOTAL = Counter("inference_requests_total", "Total requests handled", ["status"])
    TOKENS_TOTAL = Counter("inference_tokens_total", "Total tokens processed", ["direction"])
    ERRORS_TOTAL = Counter("inference_errors_total", "Total errors", ["error_class"])

    # Custom buckets: extended out to 30s/60s because LLM tail latency under overload
    # (notebook 29b saw P99s past 10 SECONDS) blows straight through prometheus_client's
    # web-request-shaped defaults, which top out at 10s -- see the comparison cell below.
    LATENCY_BUCKETS = (0.05, 0.1, 0.25, 0.5, 1, 2.5, 5, 10, 30, 60)
    TTFT_HISTOGRAM = Histogram("inference_ttft_seconds", "Time to first token", buckets=LATENCY_BUCKETS)
    E2E_HISTOGRAM = Histogram("inference_e2e_seconds", "End-to-end request latency", buckets=LATENCY_BUCKETS)

    QUEUE_DEPTH = Gauge("inference_queue_depth", "Requests waiting for a free GPU slot")
    ACTIVE_SLOTS = Gauge("inference_active_slots", "Currently busy GPU slots")

    class GenRequest(BaseModel):
        prompt: str = ""

    @app.get("/health")
    def health():
        return {"status": "ok"}

    @app.get("/metrics")
    def metrics():
        return Response(generate_latest(), media_type=CONTENT_TYPE_LATEST)

    @app.post("/generate")
    async def generate(req: GenRequest):
        t0 = time.perf_counter()
        QUEUE_DEPTH.inc()
        try:
            async with gpu_slots:
                QUEUE_DEPTH.dec()
                ACTIVE_SLOTS.inc()
                try:
                    await asyncio.sleep(TTFT_BASE_S)
                    ttft = time.perf_counter() - t0
                    TTFT_HISTOGRAM.observe(ttft)
                    TOKENS_TOTAL.labels(direction="input").inc(len(req.prompt.split()) or 1)
                    for _ in range(N_TOKENS):
                        await asyncio.sleep(TPOT_S * random.uniform(0.8, 1.2))
                    TOKENS_TOTAL.labels(direction="output").inc(N_TOKENS)
                finally:
                    ACTIVE_SLOTS.dec()
            e2e = time.perf_counter() - t0
            E2E_HISTOGRAM.observe(e2e)
            REQUESTS_TOTAL.labels(status="success").inc()
            return {"tokens": N_TOKENS, "ttft": ttft, "e2e": e2e}
        except Exception:
            ERRORS_TOTAL.labels(error_class="internal").inc()
            REQUESTS_TOTAL.labels(status="error").inc()
            raise
''')
(APP_DIR / "app.py").write_text(APP_SOURCE)
print(f"Wrote instrumented server to {APP_DIR / 'app.py'}")


Wrote instrumented server to /tmp/nb29c_app/app.py


In [3]:
server_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app:app", "--host", "127.0.0.1", "--port", str(PORT), "--log-level", "warning"],
    cwd=str(APP_DIR), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
atexit.register(server_proc.terminate)

def wait_for_health(port, timeout_s=15):
    import urllib.request
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{port}/health", timeout=0.5)
            return True
        except Exception:
            time.sleep(0.3)
    return False

server_up = wait_for_health(PORT)
BASE_URL = f"http://127.0.0.1:{PORT}"
print(f"Server healthy: {server_up}")


Server healthy: True


## Generating some real traffic to observe

A quick burst of concurrent requests — enough to push past the server's 16-slot capacity so queueing actually shows up in the metrics, the same overload signal notebook 29b measured directly on the client side.


In [4]:
import asyncio
import httpx

async def fire_burst(base_url, n_requests, concurrency):
    limits = httpx.Limits(max_connections=concurrency + 50, max_keepalive_connections=concurrency + 50)
    async with httpx.AsyncClient(timeout=30.0, limits=limits) as client:
        sem = asyncio.Semaphore(concurrency)
        async def one():
            async with sem:
                await client.post(f"{base_url}/generate", json={"prompt": "hello world"})
        await asyncio.gather(*[one() for _ in range(n_requests)])

await fire_burst(BASE_URL, n_requests=200, concurrency=48)
print("Burst complete.")


Burst complete.


## Scraping and parsing `/metrics` by hand

Real monitoring stacks (Prometheus itself) do this scrape-and-parse step for you on an interval. Doing it by hand once removes the mystery: the exposition format is just plain text, one line per label-combination.


In [5]:
import re

def parse_prometheus_text(text):
    """Parse Prometheus exposition format into {metric_name: [(labels_dict, value), ...]}.
    Real Prometheus does exactly this on every scrape -- there's no hidden binary protocol.
    """
    metrics = {}
    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        m = re.match(r'^([a-zA-Z_:][a-zA-Z0-9_:]*)(\{(.*)\})?\s+([0-9eE+\-.]+)$', line)
        if not m:
            continue
        name, _, labelstr, value = m.groups()
        labels = {}
        if labelstr:
            for k, v in re.findall(r'(\w+)="([^"]*)"', labelstr):
                labels[k] = v
        metrics.setdefault(name, []).append((labels, float(value)))
    return metrics

resp = httpx.get(f"{BASE_URL}/metrics")
parsed = parse_prometheus_text(resp.text)
print(f"Total requests so far: {parsed['inference_requests_total']}")
print(f"Output tokens so far:  {[v for l, v in parsed['inference_tokens_total'] if l['direction']=='output']}")
print(f"Current queue depth:   {parsed['inference_queue_depth'][0][1]}")


Total requests so far: [({'status': 'success'}, 200.0)]
Output tokens so far:  [1600.0]
Current queue depth:   0.0


## Computing `rate()` by hand

A counter's raw value is nearly meaningless on its own — "1,600 total tokens" doesn't tell you if that happened over one second or one hour. `rate()` is just two scrapes and a subtraction.


In [6]:
def counter_value(parsed, name, **label_filter):
    total = 0.0
    for labels, value in parsed.get(name, []):
        if all(labels.get(k) == v for k, v in label_filter.items()):
            total += value
    return total

scrape_1 = parse_prometheus_text(httpx.get(f"{BASE_URL}/metrics").text)
t1 = time.perf_counter()

# Generate a bit more traffic so the rate has something to measure.
await fire_burst(BASE_URL, n_requests=80, concurrency=48)

scrape_2 = parse_prometheus_text(httpx.get(f"{BASE_URL}/metrics").text)
t2 = time.perf_counter()

dt = t2 - t1
output_tokens_rate = (counter_value(scrape_2, "inference_tokens_total", direction="output")
                      - counter_value(scrape_1, "inference_tokens_total", direction="output")) / dt
requests_rate = (counter_value(scrape_2, "inference_requests_total", status="success")
                 - counter_value(scrape_1, "inference_requests_total", status="success")) / dt
print(f"Over {dt:.1f}s: {requests_rate:.1f} requests/sec, {output_tokens_rate:.0f} output tokens/sec")


Over 0.8s: 95.4 requests/sec, 763 output tokens/sec


## From tokens/sec to $/hour

A token-rate counter is one multiplication away from a cost figure — the same conversion notebook 32's cost engineering leans on, just derived live from a running counter instead of a monthly invoice. Using Claude Haiku 4.5's current published pricing ($1.00 / $5.00 per million input/output tokens) as a representative rate for what this simulated throughput would cost against a real hosted model:


In [7]:
INPUT_PRICE_PER_MTOK = 1.00    # Claude Haiku 4.5, per Anthropic's published pricing
OUTPUT_PRICE_PER_MTOK = 5.00

input_tokens_rate = (counter_value(scrape_2, "inference_tokens_total", direction="input")
                     - counter_value(scrape_1, "inference_tokens_total", direction="input")) / dt

cost_per_hour = (
    input_tokens_rate * 3600 / 1e6 * INPUT_PRICE_PER_MTOK
    + output_tokens_rate * 3600 / 1e6 * OUTPUT_PRICE_PER_MTOK
)
print(f"Input tokens/sec:  {input_tokens_rate:6.1f}")
print(f"Output tokens/sec: {output_tokens_rate:6.1f}")
print(f"Projected cost at this rate: ${cost_per_hour:.2f}/hour")


Input tokens/sec:   190.7
Output tokens/sec:  762.8
Projected cost at this rate: $14.42/hour


That single derived number — cost per hour, live, from a counter you're already scraping for latency — is exactly the kind of metric that turns "the invoice surprised us" (notebook 32's opening problem) into "we saw it trending up three hours before the invoice arrived."

## Why bucket choice matters: a real comparison

`prometheus_client`'s **default** `Histogram` buckets top out at 10 seconds — reasonable for a typical web request, catastrophic for LLM inference where notebook 29b measured P99s well past 10 seconds under overload. Let's see exactly what breaks.


In [8]:
from prometheus_client import Histogram, CollectorRegistry, generate_latest
import numpy as np

default_reg = CollectorRegistry()
h_default = Histogram("comparison_seconds", "default buckets", registry=default_reg)
print(f"prometheus_client's default buckets: {Histogram.DEFAULT_BUCKETS}")

# The same shape of latencies notebook 29b measured under real overload: mostly
# healthy, with a meaningful tail well past 10 seconds.
rng = np.random.default_rng(0)
overload_latencies = np.concatenate([
    rng.uniform(0.1, 0.3, 20),     # healthy requests
    rng.uniform(11, 20, 8),         # overloaded tail, past the default ceiling
])
for lat in overload_latencies:
    h_default.observe(lat)

text = generate_latest(default_reg).decode()
lines = [l for l in text.splitlines() if "_bucket" in l]
for l in lines:
    print(l)


prometheus_client's default buckets: (0.005, 0.01, 0.025, 0.05, 0.075, 0.1, 0.25, 0.5, 0.75, 1.0, 2.5, 5.0, 7.5, 10.0, inf)
comparison_seconds_bucket{le="0.005"} 0.0
comparison_seconds_bucket{le="0.01"} 0.0
comparison_seconds_bucket{le="0.025"} 0.0
comparison_seconds_bucket{le="0.05"} 0.0
comparison_seconds_bucket{le="0.075"} 0.0
comparison_seconds_bucket{le="0.1"} 0.0
comparison_seconds_bucket{le="0.25"} 14.0
comparison_seconds_bucket{le="0.5"} 20.0
comparison_seconds_bucket{le="0.75"} 20.0
comparison_seconds_bucket{le="1.0"} 20.0
comparison_seconds_bucket{le="2.5"} 20.0
comparison_seconds_bucket{le="5.0"} 20.0
comparison_seconds_bucket{le="7.5"} 20.0
comparison_seconds_bucket{le="10.0"} 20.0
comparison_seconds_bucket{le="+Inf"} 28.0


Look at the last two lines: `le="10.0"` and `le="+Inf"`. Every one of those 8 overloaded requests — whether it took 11 seconds or 20 — lands in the same `+Inf` bucket. `histogram_quantile()` (and our own hand-rolled version below) has no way to tell a request that was *barely* over budget from one that was catastrophically slow, because the bucket boundaries themselves threw that information away at scrape time — it can never be recovered later, no matter how you query it. This is exactly why the mock server above uses **custom buckets extending to 30 and 60 seconds**: the bucket boundaries have to match the actual latency distribution of what you're serving, and "what a typical web request looks like" is the wrong prior for LLM inference.

## Computing `histogram_quantile()` by hand


In [9]:
def histogram_quantile(buckets, quantile):
    """buckets: list of (le, cumulative_count) sorted ascending by le (le=float('inf')
    for the last one). Reimplements PromQL's histogram_quantile: locate the bucket the
    quantile falls into, then linearly interpolate within it -- the same approximation
    real Prometheus makes, bounded by whatever resolution the bucket boundaries allow.
    """
    buckets = sorted(buckets, key=lambda b: b[0])
    total = buckets[-1][1]
    if total == 0:
        return float("nan")
    target = quantile * total
    prev_le, prev_count = 0.0, 0.0
    for le, count in buckets:
        if count >= target:
            if le == float("inf"):
                return prev_le  # can't interpolate past the last finite bucket -- this IS the information loss
            if count == prev_count:
                continue
            frac = (target - prev_count) / (count - prev_count)
            return prev_le + frac * (le - prev_le)
        prev_le, prev_count = le, count
    return buckets[-1][0]

ttft_buckets = [(float(l["le"]) if l["le"] != "+Inf" else float("inf"), v)
                for l, v in scrape_2["inference_ttft_seconds_bucket"]]
for q in [0.5, 0.9, 0.99]:
    est = histogram_quantile(ttft_buckets, q)
    print(f"TTFT p{int(q*100)} (from histogram buckets): {est*1000:.1f}ms")


TTFT p50 (from histogram buckets): 338.0ms
TTFT p90 (from histogram buckets): 467.6ms
TTFT p99 (from histogram buckets): 496.8ms


## Shipping the real thing

The pieces above are enough to build a genuine Prometheus + Grafana stack on your own machine (this environment has no Docker daemon, so these files are written and *validated for correctness* here, not run). Point Prometheus at any server exposing `/metrics` in this same format — including a real vLLM or SGLang server, which ship these exact metric names — and the queries below work unchanged.


In [10]:
import json
import yaml
from pathlib import Path

DEPLOY_DIR = Path("deploy/observability")
DEPLOY_DIR.mkdir(parents=True, exist_ok=True)

docker_compose = {
    "services": {
        "prometheus": {
            "image": "prom/prometheus:latest",
            "ports": ["9090:9090"],
            "volumes": ["./prometheus.yml:/etc/prometheus/prometheus.yml"],
        },
        "grafana": {
            "image": "grafana/grafana:latest",
            "ports": ["3000:3000"],
            "volumes": ["./grafana-dashboard.json:/var/lib/grafana/dashboards/inference.json"],
            "depends_on": ["prometheus"],
        },
    }
}
(DEPLOY_DIR / "docker-compose.yml").write_text(yaml.safe_dump(docker_compose, sort_keys=False))

prometheus_config = {
    "global": {"scrape_interval": "5s"},
    "scrape_configs": [
        {"job_name": "inference-server", "static_configs": [{"targets": ["host.docker.internal:8000"]}]}
    ],
}
(DEPLOY_DIR / "prometheus.yml").write_text(yaml.safe_dump(prometheus_config, sort_keys=False))

# Validate both round-trip through the real YAML parser before trusting them.
for f in ["docker-compose.yml", "prometheus.yml"]:
    yaml.safe_load((DEPLOY_DIR / f).read_text())
    print(f"{f}: valid YAML, {(DEPLOY_DIR / f).stat().st_size} bytes")


docker-compose.yml: valid YAML, 344 bytes
prometheus.yml: valid YAML, 138 bytes


In [11]:
grafana_dashboard = {
    "title": "Inference Server",
    "panels": [
        {
            "title": "TTFT p95 / p99",
            "type": "timeseries",
            "targets": [
                {"expr": 'histogram_quantile(0.95, rate(inference_ttft_seconds_bucket[5m]))'},
                {"expr": 'histogram_quantile(0.99, rate(inference_ttft_seconds_bucket[5m]))'},
            ],
        },
        {
            "title": "Tokens / sec",
            "type": "timeseries",
            "targets": [{"expr": 'rate(inference_tokens_total{direction="output"}[5m])'}],
        },
        {
            "title": "Error rate",
            "type": "timeseries",
            "targets": [{"expr": 'rate(inference_requests_total{status="error"}[5m]) / rate(inference_requests_total[5m])'}],
        },
        {
            "title": "Cost per hour ($)",
            "type": "stat",
            "targets": [{"expr": 'rate(inference_tokens_total{direction="output"}[5m]) * 3600 / 1e6 * 5.0'}],
        },
    ],
}
(DEPLOY_DIR / "grafana-dashboard.json").write_text(json.dumps(grafana_dashboard, indent=2))

# Validate it round-trips as JSON and every panel has the fields Grafana requires.
loaded = json.loads((DEPLOY_DIR / "grafana-dashboard.json").read_text())
assert all("title" in p and "targets" in p for p in loaded["panels"])
print(f"grafana-dashboard.json: valid JSON, {len(loaded['panels'])} panels")


grafana-dashboard.json: valid JSON, 4 panels


## Exercises


In [12]:
# Exercise 1 (Warm-up): Recreate the default-bucket information loss for E2E latency
# Task: Build a fresh CollectorRegistry with an E2E histogram using DEFAULT buckets, feed
#       it the notebook 29b-style overload latencies (mostly 0.1-0.3s, a tail from 15-25s),
#       and print the bucket counts. What fraction of requests end up indistinguishable in
#       the +Inf bucket? Then rebuild it with buckets extended to (30, 60) and show the tail
#       is now resolved into two separate buckets instead of one.
# Hint: Reuse the `overload_latencies` construction pattern from the comparison cell above,
#       just with a wider/longer tail.

# YOUR CODE HERE


In [13]:
# Exercise 2 (Apply): Add a cost-per-request metric and a Prometheus alert rule
# Task, part A: Using scrape_1/scrape_2 and the counters already in `parsed`, write
#       cost_per_request(scrape_a, scrape_b, dt) that returns $/request (NOT $/hour) --
#       divide the $/hour figure by the requests/sec rate over the same window instead of
#       by 3600, so it answers "what does one typical request cost" rather than "what's our
#       hourly burn."
# Task, part B: Write a Prometheus alerting rule (as a Python dict, matching real
#       Prometheus alert-rule YAML shape: groups -> rules -> alert/expr/for/labels/
#       annotations) that fires when projected cost exceeds $50/hour for 5 minutes
#       straight, using the same `rate(...) * 3600 / 1e6 * price` expression as the
#       Grafana panel above. Write it to deploy/observability/alerts.yml and validate
#       it parses with yaml.safe_load.
# Hint: cost_per_request = cost_per_hour / (requests_rate * 3600) if requests_rate else 0.

# YOUR CODE HERE


In [14]:
# Exercise 3 (Extend): Connect goodput to an alert, and to autoscaling
# Task: Notebook 29b defined goodput as the fraction of requests meeting a TTFT SLO. Using
#       parsed['inference_ttft_seconds_bucket'], compute an approximate goodput for a
#       300ms SLO directly from the histogram (NOT the raw per-request TTFTs, which a real
#       Prometheus-based system never has -- only the bucketed counts). Compare your
#       histogram-based estimate to what you'd get if you had the raw values, and note why
#       a real production system trades exact accuracy for the enormously smaller storage
#       footprint of buckets. Then write ONE sentence connecting this metric to notebook
#       29d: which of Prometheus's metric types (Counter, Histogram, or Gauge) would a
#       Kubernetes autoscaler actually want to scale on, and why not the others?
# Hint: goodput_estimate = count_at_bucket(0.25 or 0.5, whichever your buckets have closest
#       to 0.3) / total_count -- you won't have an EXACT le=0.3 bucket, so estimate from
#       the closest bucket boundary and say so explicitly rather than silently interpolating.

# YOUR CODE HERE


<details>
<summary>Show solutions</summary>

```python
# Exercise 1
default_reg2 = CollectorRegistry()
h2 = Histogram("ex1_default_seconds", "default buckets", registry=default_reg2)
ex1_latencies = np.concatenate([rng.uniform(0.1, 0.3, 20), rng.uniform(15, 25, 8)])
for lat in ex1_latencies:
    h2.observe(lat)
text2 = generate_latest(default_reg2).decode()
for l in text2.splitlines():
    if "_bucket" in l:
        print(l)
# 8/28 requests (~29%) land in +Inf with default buckets -- completely indistinguishable
# whether they took 15s or 25s.

custom_reg2 = CollectorRegistry()
h3 = Histogram("ex1_custom_seconds", "custom buckets", registry=custom_reg2,
               buckets=(0.05, 0.1, 0.25, 0.5, 1, 2.5, 5, 10, 30, 60))
for lat in ex1_latencies:
    h3.observe(lat)
text3 = generate_latest(custom_reg2).decode()
for l in text3.splitlines():
    if "_bucket" in l:
        print(l)
# With buckets extended to 30/60, the tail resolves into le=30.0 (all 8) vs le=60.0 (still
# all 8, since none exceeded 30s here) -- still coarse, but no longer a single catch-all
# that erases every overloaded request's actual severity.

# Exercise 2, part A
def cost_per_request(scrape_a, scrape_b, dt, input_price=1.00, output_price=5.00):
    in_rate = (counter_value(scrape_b, "inference_tokens_total", direction="input")
               - counter_value(scrape_a, "inference_tokens_total", direction="input")) / dt
    out_rate = (counter_value(scrape_b, "inference_tokens_total", direction="output")
                - counter_value(scrape_a, "inference_tokens_total", direction="output")) / dt
    req_rate = (counter_value(scrape_b, "inference_requests_total", status="success")
                - counter_value(scrape_a, "inference_requests_total", status="success")) / dt
    hourly_cost = in_rate * 3600 / 1e6 * input_price + out_rate * 3600 / 1e6 * output_price
    return hourly_cost / (req_rate * 3600) if req_rate else 0.0

print(f"Cost per request: ${cost_per_request(scrape_1, scrape_2, dt):.5f}")

# Exercise 2, part B
alert_rules = {
    "groups": [
        {
            "name": "inference_cost_alerts",
            "rules": [
                {
                    "alert": "HighCostPerHour",
                    "expr": 'rate(inference_tokens_total{direction="output"}[5m]) * 3600 / 1e6 * 5.0 > 50',
                    "for": "5m",
                    "labels": {"severity": "warning"},
                    "annotations": {"summary": "Inference cost trending above $50/hour"},
                }
            ],
        }
    ]
}
(DEPLOY_DIR / "alerts.yml").write_text(yaml.safe_dump(alert_rules, sort_keys=False))
yaml.safe_load((DEPLOY_DIR / "alerts.yml").read_text())
print("alerts.yml: valid YAML")

# Exercise 3
def count_at_bucket(bucket_data, le):
    for labels, value in bucket_data:
        if labels["le"] == str(le) or (le == float("inf") and labels["le"] == "+Inf"):
            return value
    return None

closest_bucket = 0.25  # our buckets don't have exactly 0.3; 0.25 is the nearest below it
below = count_at_bucket(scrape_2["inference_ttft_seconds_bucket"], closest_bucket)
total = count_at_bucket(scrape_2["inference_ttft_seconds_bucket"], float("inf"))
print(f"Goodput estimate (TTFT <= {closest_bucket}s, nearest bucket to a 300ms SLO): {below/total:.1%}")
# This is an ESTIMATE bounded by bucket resolution, not the exact fraction under 300ms --
# a real system accepts that approximation because storing a histogram is O(num_buckets)
# regardless of request volume, while storing every raw TTFT would grow without bound.
#
# For notebook 29d: a Kubernetes autoscaler wants a GAUGE -- specifically something like
# queue depth or active-slot utilization, because a Gauge reports the CURRENT state to
# scale against right now. A Counter's raw value is meaningless for a scaling decision
# (you'd have to compute a rate first, adding lag); a Histogram tells you about latency
# distribution, which is a symptom of being under-scaled, not a direct signal of how much
# MORE capacity is needed the way a queue-depth Gauge is.
```
</details>


In [15]:
# Cleanup
server_proc.terminate()
try:
    server_proc.wait(timeout=5)
except subprocess.TimeoutExpired:
    server_proc.kill()
print("Server terminated.")


Server terminated.


## Key Takeaways
- **Counter, Histogram, Gauge** aren't interchangeable — a Counter only goes up (compute `rate()`, never read it raw), a Histogram approximates percentiles from bucketed counts, a Gauge reports current state directly. Picking the wrong type for a metric silently breaks the queries you'll want to run on it later.
- **Histogram bucket boundaries are a real design decision**, not boilerplate: `prometheus_client`'s defaults top out at 10 seconds, and any LLM request slower than that becomes indistinguishable from one 10x slower — the information is gone at scrape time and no query can recover it.
- `rate()` and `histogram_quantile()` are not magic — they're a subtraction-over-time and a linear interpolation within a bucket, both simple enough to hand-roll and verify against ground truth.
- A token-rate counter is one multiplication away from a live **$/hour** figure — the same cost-engineering idea from notebook 32, but observable in real time instead of discovered on next month's invoice.
- The `docker-compose.yml`, `prometheus.yml`, and Grafana dashboard JSON in `deploy/observability/` are the real artifacts — validated for correctness here, runnable as-is on any machine with Docker, and they point at the exact metric names a real vLLM or SGLang server already exposes.

## What's Next
Notebook **29d — Kubernetes for AI Workloads** takes the Gauge metrics this notebook just built (queue depth, active slots) and uses them to drive an actual autoscaling decision — the answer to Exercise 3's last question, worked out in full.
